# UTS Data Mining II — TEXT PROCESSING & K-MEANS

Materi dari modul: tokenisasi, stopword, stemming (Sastrawi), TextBlob, frekuensi kata/WordCloud, representasi TF-IDF, clustering K-Means dan evaluasi internal. Data: **IndoNLU SmSA**, kumpulan ulasan bahasa Indonesia. Label sentimen dipakai hanya untuk melihat distribusi setelah clustering, bukan untuk melatih K-Means.

**Petunjuk:** Buka di Google Colab → `Runtime` → `Run all`, atau jalankan dari atas secara berurutan. Setiap sel **kode** menghasilkan output. Semua angka hasil yang dibahas pada catatan setelah sel adalah panduan interpretasi, **bukan hasil yang diasumsikan sudah diperoleh**.

## Panduan awal — membuka berbagai file

| Jenis file | Perintah paling singkat | Keterangan |
|---|---|---|
| CSV | `pd.read_csv('file.csv')` | Tabel dengan pemisah koma |
| TSV | `pd.read_csv('file.tsv', sep='\t')` | Tabel dengan pemisah tab |
| Excel | `pd.read_excel('file.xlsx')` | Mungkin perlu `openpyxl` |
| JSON | `pd.read_json('file.json')` | Data JSON yang cocok untuk tabel |
| TXT | `open('file.txt', encoding='utf-8').read()` | Teks bebas; tidak harus melalui pandas |
| Gambar | `PIL.Image.open('foto.jpg')` atau `cv2.imread(...)` | **Bukan** dibuka langsung dengan pandas |
| Audio | `librosa.load('audio.wav')` atau `sf.read(...)` | **Bukan** dibuka langsung dengan pandas |

Pandas terutama dipakai untuk data berbentuk tabel dan untuk menampung fitur yang sudah diekstraksi dari teks/gambar/audio. Dataset latihan di notebook ini diakses langsung tanpa perlu upload manual (koneksi internet diperlukan untuk unduhan pertama).

## Contoh tabel pandas

In [ ]:
import pandas as pd
from IPython.display import display

# Contoh objek DataFrame; .head() untuk melihat baris pertama.
contoh = pd.DataFrame({'jenis': ['Text', 'Image', 'Audio'],
                       'fitur': ['TF-IDF', 'Pixel', 'MFCC']})
display(contoh)
print('Ukuran tabel:', contoh.shape)

**Interpretasi:** Output menunjukkan tiga jenis data beserta contoh fitur numeriknya. Untuk pemeriksaan tabel gunakan `.head()`, `.shape`, `.info()`, `.isna().sum()`.

**Keyword:** `pandas read_csv sep tab`, `pandas head shape info`, `read_excel google colab`.

## 1. Install library tambahan

In [ ]:
%pip -q install Sastrawi textblob wordcloud
print('Selesai menginstal Sastrawi, TextBlob, dan WordCloud.')

**Interpretasi:** `pip` memasang paket yang belum tersedia. `Sastrawi` adalah paket Python untuk bahasa Indonesia; `TextBlob` dipakai sebagai alat NLP yang utamanya berorientasi bahasa Inggris. **Keyword:** `pip install Sastrawi Colab`, `TextBlob install Python`, `wordcloud package`.

## 2. Import library

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from textblob import TextBlob
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import silhouette_score, davies_bouldin_score
from wordcloud import WordCloud
import nltk

# NLTK adalah library utama untuk sumber tokenizer yang digunakan TextBlob.
print('Paket NLTK punkt_tab berhasil diunduh:', nltk.download('punkt_tab', quiet=True))
print('Import selesai. Import from berarti mengambil class tertentu dari suatu library.')

**Interpretasi:** `sklearn` = library utama scikit-learn, sedangkan `KMeans`, `TfidfVectorizer`, dan `silhouette_score` adalah class/fungsi di submodulnya. `TextBlob` berasal dari `textblob`; `StemmerFactory` berasal dari `Sastrawi`. **Keyword:** `from sklearn.cluster import KMeans`, `TextBlob MissingCorpusError punkt_tab`, `Sastrawi StemmerFactory import`.

## 3. Ambil dataset ulasan Indonesia dari URL

In [ ]:
# URL IndoNLU / SmSA: file TSV tanpa header.
url = 'https://raw.githubusercontent.com/IndoNLP/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv'
df_asli = pd.read_csv(url, sep='\t', header=None, names=['teks', 'sentimen'])
df_asli = df_asli.dropna(subset=['teks']).reset_index(drop=True)
print('Ukuran dataset asal:', df_asli.shape)
display(df_asli.head(5))

**Interpretasi:** Setiap **baris** merupakan satu dokumen/ulasan. Dataset berlabel tetapi K-Means tetap **unsupervised** karena label tidak dipakai sebagai fitur. Jika URL gagal, periksa koneksi internet Colab. **Keyword:** `pandas read_csv url tsv header None names`, `IndoNLU SmSA Github train_preprocess.tsv`.

## 4. Sampling 120 data

In [ ]:
# Ubah angka 120 menjadi 100, 300, 500, atau 1000 sesuai kebutuhan.
n_sampel = 120
df = df_asli.sample(n=min(n_sampel, len(df_asli)), random_state=42).reset_index(drop=True)
print('Setelah sampling:', df.shape)
print('Distribusi label (hanya informasi):')
print(df['sentimen'].value_counts())
display(df[['teks', 'sentimen']].head(5))

**Interpretasi:** `.sample()` memilih **baris/dokumen**, bukan kata. `random_state=42` membuat pilihan data konsisten. **Keyword:** `pandas sample n random_state`, `sample 100 rows pandas`, `pandas value_counts`.

## 5. Case folding dan cleaning sederhana

In [ ]:
# Huruf kecil, hapus simbol/angka, lalu rapikan spasi.
df['bersih'] = (df['teks'].astype(str).str.lower()
                .str.replace(r'[^a-z\s]', ' ', regex=True)
                .str.replace(r'\s+', ' ', regex=True).str.strip())
display(df[['teks', 'bersih']].head(4))
print('Jumlah dokumen kosong:', (df['bersih'] == '').sum())

**Interpretasi:** **Case folding** menyamakan huruf kapital/kecil; **cleaning** membuang karakter lain. Pola `[a-z]` cocok untuk contoh teks Latin ini, namun angka, tanda baca, dan beberapa informasi dapat hilang. **Keyword:** `python regex remove punctuation`, `pandas str replace regex lowercase`, `text preprocessing case folding`.

## 6. Contoh perbedaan Sastrawi vs TextBlob

In [ ]:
stemmer = StemmerFactory().create_stemmer()
stopword_id = set(StopWordRemoverFactory().get_stop_words())

contoh_id = 'Mahasiswa mempelajari pembelajaran dan menjalankan penelitian.'
contoh_en = 'Children are running with their friends.'
print('Asli Indonesia :', contoh_id)
print('Sastrawi stem  :', stemmer.stem(contoh_id.lower()))
print('TextBlob token :', list(TextBlob(contoh_en).words))
print('Stopword Sastrawi (contoh):', sorted(list(stopword_id))[:15])

**Interpretasi:** **Sastrawi** melakukan stemming bahasa Indonesia (`mempelajari → ajar` biasanya), sedangkan **TextBlob** menyediakan tokenisasi/fitur NLP dan umumnya lebih sesuai untuk **bahasa Inggris**. Dalam notebook ini, TextBlob digunakan untuk **tokenisasi teks Indonesia**, bukan lemmatization Indonesia (tidak tersedia secara andal). **Keyword:** `Sastrawi stem stemming Indonesian`, `TextBlob words tokenize`, `difference stemming lemmatization TextBlob`.

## 7. Pipeline A — preprocessing Sastrawi

In [ ]:
# Gunakan stemmer Indonesia dan buang stopword.
df['teks_sastrawi'] = [
    ' '.join(kata for kata in stemmer.stem(teks).split()
             if kata not in stopword_id and len(kata) > 2)
    for teks in df['bersih']
]
display(df[['bersih', 'teks_sastrawi']].head(5))
print('Contoh jumlah token:', len(df.loc[0, 'teks_sastrawi'].split()))

**Interpretasi:** Kata berimbuhan disederhanakan ke **kata dasar**, kemudian stopword dibuang. Pada data lebih besar proses stemming lebih lambat; bila terburu-buru kurangi `n_sampel` pada tahap 4. **Keyword:** `Sastrawi stemmer.stem example`, `Sastrawi stopword remover factory`, `python list comprehension text preprocessing`.

## 8. WordCloud dan barplot kata terbanyak

In [ ]:
frekuensi = Counter(' '.join(df['teks_sastrawi']).split())
teratas = pd.DataFrame(frekuensi.most_common(12), columns=['kata', 'frekuensi'])
fig, ax = plt.subplots(1, 2, figsize=(14, 5))
wc = WordCloud(width=700, height=350, background_color='white').generate_from_frequencies(frekuensi)
ax[0].imshow(wc)
ax[0].axis('off')
ax[0].set_title('WordCloud - Sastrawi')
ax[1].barh(teratas['kata'][::-1], teratas['frekuensi'][::-1])
ax[1].set_title('12 kata paling sering')
ax[1].set_xlabel('Frekuensi')
plt.tight_layout()
plt.show()
display(teratas)

**Interpretasi:** Kata terbesar pada WordCloud atau barplot = kata yang paling sering muncul **setelah preprocessing**, bukan otomatis kata yang paling penting secara semantik. **Keyword:** `WordCloud generate_from_frequencies Counter`, `python most common words barplot`, `matplotlib barh`.

## 9. TF-IDF dan K-Means pipeline Sastrawi

In [ ]:
# X berupa matriks (jumlah dokumen x jumlah kata/fitur).
vec_s = TfidfVectorizer(min_df=2, max_features=1000)
X_s = vec_s.fit_transform(df['teks_sastrawi'])
print('Shape TF-IDF Sastrawi:', X_s.shape)

km_s = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_sastrawi'] = km_s.fit_predict(X_s)
print('Banyak anggota cluster:')
print(df['cluster_sastrawi'].value_counts().sort_index())

kata_s = vec_s.get_feature_names_out()
for i in range(3):
    idx = km_s.cluster_centers_[i].argsort()[-8:][::-1]
    print('Cluster', i, ':', ', '.join(kata_s[idx]))

**Interpretasi:** TF-IDF mengubah dokumen menjadi **vektor numerik**. `KMeans(n_clusters=3)` membentuk tiga cluster **tanpa memakai label sentimen**. Periksa kata-kata dengan bobot centroid tertinggi untuk menafsirkan topik cluster. Nomor cluster tidak mempunyai arti urutan. **Keyword:** `TfidfVectorizer fit_transform shape`, `KMeans fit_predict sparse tfidf`, `top terms cluster center kmeans`.

## 10. Pipeline B — TextBlob tokenizer pada teks yang sama

In [ ]:
# TextBlob digunakan untuk tokenisasi, bukan stemming/lemma Indonesia.
df['teks_textblob'] = [
    ' '.join(str(kata).lower() for kata in TextBlob(teks).words
             if str(kata).lower() not in stopword_id and len(str(kata)) > 2)
    for teks in df['bersih']
]
display(df[['teks_sastrawi', 'teks_textblob']].head(6))
print('Jumlah dokumen:', len(df))

**Interpretasi:** Hasil TextBlob mempertahankan kata berimbuhan karena contoh ini hanya **tokenization + stopword removal**, sedangkan pipeline Sastrawi menggunakan **stemming**. Jadi hasilnya membandingkan dua **pipeline preprocessing**, bukan kinerja dua stemmer bahasa Indonesia. **Keyword:** `TextBlob Words words tokenization`, `TextBlob bahasa Indonesia stemming`, `TextBlob tokenizer nltk punkt_tab`.

## 11. TF-IDF dan K-Means pipeline TextBlob

In [ ]:
vec_b = TfidfVectorizer(min_df=2, max_features=1000)
X_b = vec_b.fit_transform(df['teks_textblob'])
print('Shape TF-IDF TextBlob:', X_b.shape)

km_b = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster_textblob'] = km_b.fit_predict(X_b)
print('Banyak anggota cluster:')
print(df['cluster_textblob'].value_counts().sort_index())

kata_b = vec_b.get_feature_names_out()
for i in range(3):
    idx = km_b.cluster_centers_[i].argsort()[-8:][::-1]
    print('Cluster', i, ':', ', '.join(kata_b[idx]))

**Interpretasi:** Tahapan **TF-IDF → K-Means** sama seperti pada Sastrawi, tetapi representasi kata yang masuk berbeda. Ukuran TF-IDF bisa berbeda karena kosakata hasil cleaning berbeda. **Keyword:** `TextBlob sklearn TF-IDF KMeans`, `how to cluster Indonesian text with sklearn`.

## 12. Evaluasi kedua hasil clustering

In [ ]:
perbandingan = pd.DataFrame({
    'pipeline': ['Sastrawi', 'TextBlob tokenizer'],
    'silhouette': [silhouette_score(X_s, df['cluster_sastrawi']),
                   silhouette_score(X_b, df['cluster_textblob'])],
    'davies_bouldin': [davies_bouldin_score(X_s.toarray(), df['cluster_sastrawi']),
                       davies_bouldin_score(X_b.toarray(), df['cluster_textblob'])]
})
display(perbandingan)
print('Distribusi label referensi vs cluster Sastrawi:')
display(pd.crosstab(df['sentimen'], df['cluster_sastrawi']))

**Interpretasi:** **Silhouette** lebih tinggi (mendekati 1) biasanya lebih baik; **Davies–Bouldin** lebih rendah biasanya lebih baik. Nilai kedua pipeline dipengaruhi perbedaan representasi fitur. Crosstab di sini **bukan** akurasi klasifikasi; label hanya informasi evaluasi eksternal. **Keyword:** `silhouette_score TFIDF KMeans`, `davies_bouldin_score sparse toarray`, `cluster crosstab true label`.

## 13. Visualisasi cluster 2D (TruncatedSVD)

In [ ]:
pos_s = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_s)
pos_b = TruncatedSVD(n_components=2, random_state=42).fit_transform(X_b)
print('Shape SVD Sastrawi:', pos_s.shape)
print('Shape SVD TextBlob:', pos_b.shape)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].scatter(pos_s[:, 0], pos_s[:, 1], c=df['cluster_sastrawi'], cmap='viridis', s=22)
ax[0].set_title('Sastrawi + K-Means')
ax[1].scatter(pos_b[:, 0], pos_b[:, 1], c=df['cluster_textblob'], cmap='viridis', s=22)
ax[1].set_title('TextBlob token + K-Means')
for a in ax:
    a.set_xlabel('Komponen 1')
    a.set_ylabel('Komponen 2')
plt.tight_layout()
plt.show()

**Interpretasi:** SVD mereduksi dimensi agar cluster bisa **ditampilkan** pada bidang 2D; K-Means sebelumnya berjalan di ruang TF-IDF aslinya. Bila titik terlihat tumpang tindih, belum tentu seluruh cluster gagal karena ada informasi yang hilang akibat proyeksi. **Keyword:** `TruncatedSVD tfidf 2D visualization`, `matplotlib scatter KMeans text`, `dimensionality reduction text clustering`.

## 14. Lihat contoh dokumen dari tiap cluster

In [ ]:
for c in sorted(df['cluster_sastrawi'].unique()):
    print('\nCLUSTER SASTRAWI', c)
    for kalimat in df.loc[df['cluster_sastrawi'] == c, 'teks'].head(2):
        print('-', kalimat[:180])
print('\nSelesai praktikum teks.')

**Interpretasi:** Cluster wajib ditafsirkan berdasarkan **contoh teks nyata** dan kata dominan, bukan berdasarkan nomor 0/1/2. Alur ringkas: **ambil data → sampling → cleaning → preprocessing → frekuensi → TF-IDF → K-Means → evaluasi → interpretasi**. **Keyword:** `how to interpret text clusters examples`, `extract rows by kmeans cluster pandas`.